<a href="https://colab.research.google.com/github/shin584/project/blob/model_analysis/SpCas9_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
%%writefile /content/nt_v3.py
import os; os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"   # 모델 다운로드 표시줄 끄기
import argparse, inspect, os, numpy as np, pandas as pd, torch
from scipy.stats import spearmanr
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, EarlyStoppingCallback, set_seed)

ap = argparse.ArgumentParser()
ap.add_argument("--seed", type=int, default=42); ap.add_argument("--epochs", type=int, default=5)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--data", default=os.environ.get("DATA", "/content")); ap.add_argument("--out", default=os.environ.get("OUT", "/content"))
ap.add_argument("--model", default="InstaDeepAI/nucleotide-transformer-500m-human-ref")
ap.add_argument("--sub", type=int, default=0)
a = ap.parse_args(); set_seed(a.seed); os.makedirs(a.out, exist_ok=True)

H = ['SpCas9','SpCas9-NG','VRQR variant','xCas','Sniper-Cas9','SpCas9-HF.1','eSpCas9(1.1)','HypaCas9','evoCas9']
tr = pd.read_csv(f"{a.data}/train.csv"); va = pd.read_csv(f"{a.data}/val.csv"); te = pd.read_csv(f"{a.data}/test.csv")
if a.sub: tr, va, te = tr.sample(a.sub, random_state=0), va.sample(a.sub//5, random_state=0), te.sample(a.sub//5, random_state=0)
for d in (tr, va, te): assert d.sequence.str.len().eq(30).all()
print("train", len(tr), "val", len(va), "test", len(te))

Ytr = tr[H].values.astype(np.float32) / 100.0
mu = np.nanmean(Ytr, 0); sd = np.nanstd(Ytr, 0)                 # 헤드별 표준화 (측정된 값만)
cov = (~np.isnan(Ytr)).mean(0).astype(np.float32)               # 헤드별 측정 비율 (손실 균형용)

tok = AutoTokenizer.from_pretrained(a.model)
class DS(torch.utils.data.Dataset):
    def __init__(s, d):
        s.enc = tok(list(d.sequence), padding="max_length", truncation=True, max_length=32)
        y = d[H].values.astype(np.float32) / 100.0
        m = ~np.isnan(y)
        s.lab = np.hstack([np.nan_to_num((y - mu) / sd), m.astype(np.float32)]).astype(np.float32)   # [표준화 label 9, mask 9]
    def __len__(s): return len(s.lab)
    def __getitem__(s, i):
        x = {k: torch.tensor(v[i]) for k, v in s.enc.items()}
        x["labels"] = torch.tensor(s.lab[i]); return x

class MaskedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        lab = inputs.pop("labels"); out = model(**inputs); p = out.logits
        y, m = lab[:, :9], lab[:, 9:]
        c = torch.tensor(cov, device=p.device)
        loss = (m * (y - p) ** 2 / c).sum(-1).mean()               # 결측 헤드는 손실 0
        return (loss, out) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(a.model, num_labels=9, problem_type="regression")
k = "eval_strategy" if "eval_strategy" in inspect.signature(TrainingArguments.__init__).parameters else "evaluation_strategy"
args = TrainingArguments(output_dir=f"/content/out_ntv3_{a.seed}", num_train_epochs=a.epochs,
    per_device_train_batch_size=32, per_device_eval_batch_size=64, learning_rate=a.lr, weight_decay=0.01,
    fp16=torch.cuda.is_available(), save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False, logging_steps=200, report_to="none",
    disable_tqdm=True, seed=a.seed, **{k: "epoch"})
dtr, dva, dte = DS(tr), DS(va), DS(te)
t = MaskedTrainer(model=model, args=args, train_dataset=dtr, eval_dataset=dva,
                  callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
t.train()

def predict(ds): return t.predict(ds).predictions * sd + mu    # 원래 척도(/100)로 복원, (N,9)
def evaluate(P, d):
    y = d[H].values / 100.0; ym = ~np.isnan(y)
    gg = (d.sequence.str[25:27].values == "GG"); r = {}; a_, g_ = [], []
    for j in range(9):
        ok = ym[:, j]
        a_.append(spearmanr(P[ok, j], y[ok, j])[0])
        g = ok & gg; g_.append(spearmanr(P[g, j], y[g, j])[0] if g.sum() >= 50 else np.nan)
        if j == 0: r["SpCas9_MAE"] = np.abs(P[ok, 0] - y[ok, 0]).mean()
    r.update(Sp_all=a_[0], Sp_ngg=g_[0], mean_all=np.nanmean(a_), mean_ngg=np.nanmean(g_))
    return r
Pva, Pte = predict(dva), predict(dte)
rv, rt = evaluate(Pva, va), evaluate(Pte, te)
row = dict(cfg="NT_v3", seed=a.seed, val_Sp_ngg=rv["Sp_ngg"], **{"test_" + k_: v for k_, v in rt.items()})
print({k_: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k_, v in row.items()})
pd.DataFrame([row]).to_csv(f"{a.out}/nt_v3_seed{a.seed}_results.csv", index=False)

M = f"{a.out}/nt_v3_seed{a.seed}"                               # 모델 + 토크나이저 + 복원용 mu/sd 저장
t.save_model(M); tok.save_pretrained(M); np.save(f"{M}/mu_sd.npy", np.stack([mu, sd]))
print("저장 완료:", M)

Overwriting /content/nt_v3.py


In [6]:
!python /content/nt_v3.py --seed 42 --data "/content/drive/MyDrive/SpCas9" --out "/content/drive/MyDrive/SpCas9/spcas9_v3"

train 29134 val 3676 test 5186
[transformers] EsmForSequenceClassification LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.decoder.weight     | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
{'loss': '5.605', 'grad_norm': '37.23', 'learning_rate': '1.913e-05', 'epoch': '0.2195'}
{'lo

In [8]:
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

M = "/content/drive/MyDrive/SpCas9/spcas9_v3/nt_v3_seed42"
H = ['SpCas9','SpCas9-NG','VRQR variant','xCas','Sniper-Cas9',
     'SpCas9-HF.1','eSpCas9(1.1)','HypaCas9','evoCas9']

tok = AutoTokenizer.from_pretrained(M)
model = AutoModelForSequenceClassification.from_pretrained(M).eval().cuda()
mu, sd = np.load(f"{M}/mu_sd.npy")          # 헤드별 평균/표준편차 (9개씩)

def predict(seqs):
    """30bp 서열 리스트 -> (N, 9) 효율 (0~1 척도, 즉 %/100)"""
    out = []
    for i in range(0, len(seqs), 64):
        enc = tok(seqs[i:i+64], padding="max_length", truncation=True,
                  max_length=32, return_tensors="pt").to("cuda")
        with torch.no_grad():
            out.append(model(**enc).logits.float().cpu().numpy())
    return np.concatenate(out) * sd + mu     # 표준화 되돌리기

p = predict(["ACGT"+"ACGT"*5+"AGG"+"TCA"])
print(dict(zip(H, p[0].round(3))))

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

{'SpCas9': np.float32(0.502), 'SpCas9-NG': np.float32(0.369), 'VRQR variant': np.float32(0.336), 'xCas': np.float32(0.367), 'Sniper-Cas9': np.float32(0.489), 'SpCas9-HF.1': np.float32(0.34), 'eSpCas9(1.1)': np.float32(0.408), 'HypaCas9': np.float32(0.434), 'evoCas9': np.float32(0.104)}
